# Importancia Gini de las variables

**Objetivo:** Importancia Gini del RF; 9 variables.

Este cuaderno contiene el código completo del experimento `plot_importance.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_completo.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'importancia_rf'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_completo.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


## 3. Bibliotecas, rutas y reproducibilidad
Estas son las importaciones, rutas de entrada/salida y semillas del experimento. Los archivos generados se guardan en la carpeta de esta ejecución.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
BASE_DIR = str(OUTPUT_DIR)
FILE_COMBINADO = str(DATA_FILES['dataset_entrenamiento_completo.csv'])
FILE_GRAFICO = str(OUTPUT_DIR / 'importancia_variables.png')


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("Generando gráfico de importancia de variables...")

if not os.path.exists(FILE_COMBINADO):
    print(f"[ERROR] No se encuentra el archivo {FILE_COMBINADO}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
# 1. Cargar datos
df = pd.read_csv(FILE_COMBINADO)

features = ['elevacion', 'pendiente', 'orientacion', 'cobertura_suelo', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

# Nombres más amigables en español para el gráfico
nombres_espanol = {
    'precip_mm': 'Precipitación (Lluvia)',
    'temp_c': 'Temperatura (°C)',
    'nbr': 'Índice de Quemado (NBR)',
    'elevacion': 'Elevación (Altitud)',
    'cobertura_suelo': 'Cobertura de Suelo',
    'ndvi': 'Índice de Vegetación (NDVI)',
    'evi': 'Índice de Vegetación (EVI)',
    'pendiente': 'Pendiente del Terreno',
    'orientacion': 'Orientación de Ladera'
}

X = df[features]
y = df['clase']

# 2. Entrenar el modelo rápido
model = RandomForestClassifier(
    n_estimators=CONFIG['rf_trees'], 
    max_depth=CONFIG['rf_depth'], 
    random_state=CONFIG['seed'], 
    n_jobs=-1
)


### Etapa 2: Ajuste del modelo
Aquí se aprende a partir de las muestras de entrenamiento. Revisa los hiperparámetros visibles y evita ajustar decisiones usando repetidamente los resultados del conjunto de prueba.

In [ ]:
model.fit(X, y)

# 3. Obtener importancia
importances = model.feature_importances_
indices = np.argsort(importances) # Orden ascendente para el gráfico de barras horizontal

# Mapear nombres a español
features_es = [nombres_espanol[features[i]] for i in indices]
importances_sorted = importances[indices] * 100 # En porcentaje

# 4. Crear el gráfico con diseño premium (Colores limpios, sin bordes innecesarios)
plt.figure(figsize=(10, 6), dpi=300)

# Estilo moderno
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Colores degradados en tonos de fuego (Naranja a Rojo)
colors = plt.cm.YlOrRd(np.linspace(0.4, 0.8, len(indices)))

bars = plt.barh(range(len(indices)), importances_sorted, color=colors, edgecolor='none', height=0.6)

# Añadir valores numéricos al final de cada barra
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.5, 
        bar.get_y() + bar.get_height()/2, 
        f'{width:.2f}%', 
        va='center', 
        ha='left', 
        fontsize=10, 
        fontweight='bold', 
        color='#333333'
    )


### Etapa 3: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

plt.yticks(range(len(indices)), features_es, fontsize=11, fontweight='semibold')
plt.xticks(fontsize=10)
plt.xlabel('Importancia Relativa (%)', fontsize=12, fontweight='bold', labelpad=10)
plt.title('Factores Principales que Afectan los Incendios en Ecuador\n(Modelo Random Forest Definitivo)', 
          fontsize=14, fontweight='bold', pad=15, color='#111111')

# Ajustar límites del eje X para dar espacio a las etiquetas de texto
plt.xlim(0, max(importances_sorted) + 8)

# Quitar bordes del gráfico para diseño limpio
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_color('#cccccc')
ax.yaxis.grid(False) # Solo cuadrícula vertical
ax.xaxis.grid(True, linestyle='--', alpha=0.6, color='#dddddd')

plt.tight_layout()

# Guardar gráfico
plt.savefig(FILE_GRAFICO, bbox_inches='tight')
mostrar_figura(FILE_GRAFICO)
plt.close()

print(f"[OK] Gráfico guardado exitosamente en: {FILE_GRAFICO}")


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
